# Watem NB01 — Site selection and baseline run
No Google Drive. Select an area, inspect the watershed, run the baseline and download one portable ZIP. Keep that ZIP on your PC for NB02 and NB03. Pilot parameters, not calibrated predictions.

In [ ]:
from pathlib import Path
import os, sys, shutil, subprocess, json, hashlib, time, platform
from datetime import datetime, timezone
ROOT = Path('/content/Watem_local')
DATA = ROOT / 'data/tutorial/langegracht'
OUTPUT = ROOT / 'outputs/student_preparation'
SOFTWARE = ROOT / 'software'
for p in (DATA.parent, OUTPUT, SOFTWARE): p.mkdir(parents=True, exist_ok=True)
REV = '226a7c06ae836f9736c1a88b74b5346033a2a8d6'
REPO = Path('/content/pywatemsedem')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
WORK = Path('/content/watem_condroz_runs') / RUN_ID
WORK.mkdir(parents=True)
def run(cmd, **kwargs):
    print('+', ' '.join(map(str, cmd)), flush=True)
    return subprocess.run(list(map(str,cmd)), check=True, **kwargs)
print(ROOT, sys.version, platform.machine())
SITE_ID='my_catchment' #@param {type:'string'}
assert SITE_ID and SITE_ID.replace('_','').replace('-','').isalnum()
SITE=ROOT/'student_sites'/f'{SITE_ID}_{RUN_ID}'
assert not (SITE/'baseline'/'complete.json').exists(), 'Site already frozen: use another SITE_ID or go to NB02'
PACKAGE=SITE/'reference'
PACKAGE.mkdir(parents=True,exist_ok=True)
SEED=2271; RES=20; YEAR=2025
# local search window near Ciney; bounds chosen to stay in one DEM tile
SEARCH_LONLAT=(5.00,50.25,5.16,50.36)
TARGET_HA=250; TOL_HA=50
MIN_RIVER_CELLS=20 # >= 20 burned river pixels; not a physical length estimate
# Optional locally cached numerical SPW raster paths; do not use rendered WMS/MapServer imagery.
SPW_DEM=None
SPW_WALOUS=None # requires explicit class crosswalk; not implemented here
run([sys.executable,'-m','pip','install','pysheds','ipyleaflet'])

## 1. Choose an approximate outlet or search area
This first selection only locates the terrain download. You will confirm the exact hydrological outlet on a second map after D8 accumulation is calculated.

In [ ]:
from google.colab import output
output.enable_custom_widget_manager()
from ipyleaflet import Map, DrawControl, TileLayer, LayersControl, ScaleControl, FullScreenControl, GeoJSON, Polyline
import ipywidgets as widgets
from IPython.display import display
from shapely.geometry import shape, mapping, box, Point
import requests, unicodedata, math
selection={}
REST='https://geoservices.wallonie.be/arcgis/rest/services'
WMS='https://geoservices.wallonie.be/arcgis/services'
def service_json(path):
    r=requests.get(REST+'/'+path+'/MapServer',params={'f':'json'},timeout=45); r.raise_for_status()
    j=r.json(); assert 'layers' in j,j; return j
def norm(s):
    return ''.join(c for c in unicodedata.normalize('NFKD',s.lower()) if not unicodedata.combining(c))
picc=service_json('TOPOGRAPHIE/PICC_VDIFF')
def picc_id(*terms):
    matches=[l['id'] for l in picc['layers'] if l.get('subLayerIds') is None and all(t in norm(l['name']) for t in terms)]
    assert len(matches)==1,(terms,matches); return matches[0]
PICC_IDS=dict(river=picc_id('hydrographie','axe'),water=picc_id('hydrographie','emprise'),roads=picc_id('voirie','axe') if False else next(l['id'] for l in picc['layers'] if norm(l['name']).strip()=='voirie - axe'),buildings=picc_id('construction','batiment','emprise'))
RIVER_URL=REST+'/EAU/RES_HYDRO_WAL/MapServer/1'
print('Current PICC layer IDs:',PICC_IDS,'| river source: RHW axes')
# Cached basemaps: no WMS rendering while navigating.
ESRI='https://server.arcgisonline.com/ArcGIS/rest/services/'
def tiles(service,name,max_native=19):
    return TileLayer(url=ESRI+service+'/MapServer/tile/{z}/{y}/{x}',name=name,attribution='Esri and contributing data providers',max_zoom=20,max_native_zoom=max_native)
imagery=tiles('World_Imagery','Satellite imagery')
street=tiles('World_Street_Map','Street map')
topo=tiles('World_Topo_Map','Topographic map')
labels=tiles('Reference/World_Boundaries_and_Places','Place names',19)
roads_labels=tiles('Reference/World_Transportation','Road labels',19)
widget=Map(center=(50.36,4.9),zoom=12,layers=(topo,),scroll_wheel_zoom=True,layout=widgets.Layout(height='600px',width='100%'))
widget.add_control(ScaleControl(position='bottomleft')); widget.add_control(FullScreenControl())
river_status=widgets.HTML('<b>Waiting for map extent…</b> Rivers load automatically.')
river_progress=widgets.IntProgress(value=0,min=0,max=1,description='Rivers:',layout=widgets.Layout(width='98%'))
river_overlay=None
coverage_overlay=None
river_cache={}
active_river_data=None
active_bounds=None
river_task=None
auto_task=None
import asyncio
def line_locations(data):
    # Leaflet Polyline takes [latitude, longitude]; flatten only the feature structure.
    result=[]
    for f in data.get('features',[]):
        g=f.get('geometry') or {}
        if g.get('type')=='LineString': parts=[g['coordinates']]
        elif g.get('type')=='MultiLineString': parts=g['coordinates']
        else: continue
        result.extend([[ [float(pt[1]),float(pt[0])] for pt in part ] for part in parts if len(part)>=2])
    return result
def river_line_layer(locations):
    return Polyline(locations=locations,color='#00aaff',weight=3,opacity=1,name='RHW watercourses',pane='overlayPane',interactive=False)
def covers(bounds,extent):
    if not bounds or extent is None: return False
    (south,west),(north,east)=bounds
    return extent[0]<=west and extent[1]<=south and extent[2]>=east and extent[3]>=north
async def attach_rivers(data,extent):
    global river_overlay,coverage_overlay,active_river_data,active_bounds
    river_status.value='<b style="color:#1976d2">Rendering rivers…</b> Preparing a new browser overlay.'
    loc=await asyncio.to_thread(line_locations,data)
    if river_overlay is not None and river_overlay in widget.layers: widget.remove_layer(river_overlay)
    if coverage_overlay is not None and coverage_overlay in widget.layers: widget.remove_layer(coverage_overlay)
    await asyncio.sleep(.15)
    river_overlay=river_line_layer(loc)
    if show_rivers.value and loc: widget.add_layer(river_overlay)
    coverage_overlay=GeoJSON(data={'type':'Feature','geometry':mapping(box(*extent)),'properties':{}},name='Downloaded river extent',style={'color':'#ffad00','weight':2,'fillOpacity':0,'dashArray':'5,5'},interactive=False)
    widget.add_layer(coverage_overlay)
    active_river_data=data; active_bounds=extent
    await asyncio.sleep(.25)
    return len(loc)

from concurrent.futures import ThreadPoolExecutor, as_completed
MAX_RIVER_FEATURES=20000
def download_river_batch(ids):
    url=RIVER_URL+'/query'
    q=requests.post(url,data=dict(objectIds=','.join(map(str,ids)),outFields='*',returnGeometry='true',outSR=4326,f='geojson',geometryPrecision=6),timeout=60)
    q.raise_for_status(); q=q.json()
    assert 'features' in q and not q.get('exceededTransferLimit'),str(q)[:300]
    fs=[{'type':'Feature','geometry':f['geometry'],'properties':{}} for f in q['features'] if f.get('geometry')]
    assert len(fs)==len(ids),(len(fs),len(ids))
    return fs
async def load_rivers_async(force=False):
    global river_overlay, coverage_overlay
    load_button.disabled=True; load_button.description='Loading rivers…'; load_button.icon='spinner'
    river_status.value='<b style="color:#1976d2">Loading RHW rivers…</b> Checking the visible area.'
    river_progress.layout.display=''; river_progress.value=0; river_progress.max=1; river_progress.bar_style='info'
    await asyncio.sleep(.05)
    try:
        bounds=widget.bounds
        if not bounds: raise ValueError('Map not ready yet. Try again.')
        (south,west),(north,east)=bounds
        # No fixed zoom threshold. A feature-count safety cap prevents browser overload.
        # Smaller margins at regional zooms reduce unnecessary downloads.
        pad=.35 if widget.zoom>=12 else .10
        dx=pad*(east-west); dy=pad*(north-south)
        west-=dx; east+=dx; south-=dy; north+=dy
        key=tuple(round(v,5) for v in (west,south,east,north))
        loaded_bounds=key
        data=None
        for k,cached in river_cache.items():
            if not force and k[0]<=west and k[1]<=south and k[2]>=east and k[3]>=north:
                data=cached; loaded_bounds=k; break
        if data is None:
            url=RIVER_URL+'/query'
            params=dict(where='1=1',geometry=','.join(map(str,key)),geometryType='esriGeometryEnvelope',inSR=4326,spatialRel='esriSpatialRelIntersects',f='json',returnIdsOnly='true')
            def get_ids():
                q=requests.get(url,params=params,timeout=45); q.raise_for_status(); j=q.json()
                assert 'error' not in j,j
                return j.get('objectIds') or []
            ids=await asyncio.to_thread(get_ids)
            if len(ids)>MAX_RIVER_FEATURES:
                raise ValueError(f'This view contains {len(ids):,} stream features. Zoom in slightly (limit {MAX_RIVER_FEATURES:,}). Existing overlay retained.')
            river_progress.max=max(len(ids),1)
            river_status.value=f'<b style="color:#1976d2">Downloading {len(ids):,} river features…</b>'
            features=[]
            batches=[ids[i:i+100] for i in range(0,len(ids),100)]
            # A small worker pool speeds up local downloads without flooding the SPW service.
            loop=asyncio.get_running_loop()
            with ThreadPoolExecutor(max_workers=4) as pool:
                futures=[loop.run_in_executor(pool,download_river_batch,b) for b in batches]
                for future in asyncio.as_completed(futures):
                    features.extend(await future)
                    river_progress.value=len(features)
                    river_status.value=f'<b style="color:#1976d2">Loading rivers: {len(features):,} / {len(ids):,}</b>'
                    await asyncio.sleep(0)
            assert len(features)==len(ids),(len(features),len(ids))
            data=dict(type='FeatureCollection',features=features)
            if len(river_cache)>=6: river_cache.pop(next(iter(river_cache)))
            river_cache[key]=data
        else:
            river_status.value='<b>Using cached river data…</b>'
        count=await attach_rivers(data,loaded_bounds)
        river_progress.value=river_progress.max; river_progress.bar_style='success'
        river_status.value=f'<b style="color:#218838">✓ {len(data["features"]):,} RHW features ({count:,} lines) sent to map.</b> Auto-refresh outside the orange extent; use Refresh rivers to retry.'
    except Exception as exc:
        river_progress.bar_style='danger'
        river_status.value='River loading stopped: '+str(exc)[:350]
    finally:
        load_button.disabled=False; load_button.description='Refresh rivers'; load_button.icon='refresh'
        await asyncio.sleep(.1)
def start_rivers(force=False):
    global river_task
    if river_task is None or river_task.done():
        river_task=asyncio.ensure_future(load_rivers_async(force=force))
def load_rivers(_=None): start_rivers(force=True)
load_button=widgets.Button(description='Refresh rivers',button_style='info',icon='refresh',layout=widgets.Layout(width='160px'))
load_button.on_click(load_rivers)
show_rivers=widgets.Checkbox(value=True,description='Show rivers',indent=False)
auto_rivers=widgets.Checkbox(value=True,description='Auto-refresh',indent=False)
def toggle_rivers(change):
    if river_overlay is not None:
        if change['new'] and river_overlay not in widget.layers: widget.add_layer(river_overlay)
        elif not change['new'] and river_overlay in widget.layers: widget.remove_layer(river_overlay)
    if change['new'] and active_river_data is None: start_rivers()
show_rivers.observe(toggle_rivers,names='value')
redraw_btn=widgets.Button(description='Redraw cached',icon='refresh',layout=widgets.Layout(width='150px'))
async def redraw_cached():
    if active_river_data is None:
        start_rivers(); return
    redraw_btn.disabled=True
    try:
        count=await attach_rivers(active_river_data,active_bounds)
        river_status.value=f'<b style="color:#218838">✓ Redrawn {count:,} cached river lines.</b>'
    except Exception as exc:
        river_status.value='Redraw failed: '+str(exc)[:250]
    finally: redraw_btn.disabled=False
def redraw_overlays(_):
    if river_task is None or river_task.done(): asyncio.ensure_future(redraw_cached())
redraw_btn.on_click(redraw_overlays)
async def auto_after_pan():
    await asyncio.sleep(.8)
    if auto_rivers.value and show_rivers.value and widget.bounds and not covers(widget.bounds,active_bounds):
        if river_task is not None and not river_task.done(): await asyncio.shield(river_task)
        if auto_rivers.value and show_rivers.value and not covers(widget.bounds,active_bounds): start_rivers()
def bounds_changed(change=None):
    global auto_task
    if not auto_rivers.value or not show_rivers.value: return
    if auto_task is not None and not auto_task.done(): auto_task.cancel()
    auto_task=asyncio.ensure_future(auto_after_pan())
widget.observe(bounds_changed,names='bounds')
auto_rivers.observe(bounds_changed,names='value')
draw=DrawControl(polyline={},circle={},circlemarker={},marker={'shapeOptions':{}},polygon={'shapeOptions':{'color':'#ffbb22','fillOpacity':.12}},rectangle={'shapeOptions':{'color':'#ffbb22','fillOpacity':.12}})
status=widgets.HTML('<b>Choose the marker tool and click on a small stream.</b>')
overlay=None
def save_geom(g):
    global overlay
    selection['geometry']=g
    if overlay is not None and overlay in widget.layers: widget.remove_layer(overlay)
    overlay=GeoJSON(data={'type':'Feature','geometry':g,'properties':{}},style={'color':'#ffbb22','fillOpacity':.08})
    widget.add_layer(overlay)
    b=shape(g).bounds; status.value=f'<b>{"Outlet selected" if g["type"]=="Point" else "Search area selected"}.</b> {tuple(round(x,5) for x in b)}. Continue to next cell.'
def on_draw(target,action,geo_json):
    if action in ('created','edited'):
        if geo_json.get('type')=='FeatureCollection':
            fs=geo_json.get('features',[])
            if fs: save_geom(fs[-1]['geometry'])
        elif 'geometry' in geo_json: save_geom(geo_json['geometry'])
    elif action=='deleted':
        selection.clear()
        if overlay is not None and overlay in widget.layers: widget.remove_layer(overlay)
        status.value='Selection deleted. Choose a new outlet.'
draw.on_draw(on_draw); widget.add_control(draw)
base=widgets.ToggleButtons(options=['Topographic','Street','Satellite','Hybrid'],value='Topographic',description='Map:')
def switch(change):
    # Preserve all vector overlays and place the chosen raster(s) below them.
    vectors=tuple(l for l in widget.layers if l not in (imagery,street,topo,labels,roads_labels))
    raster={'Topographic':(topo,), 'Street':(street,), 'Satellite':(imagery,), 'Hybrid':(imagery,labels,roads_labels)}[change['new']]
    widget.layers=raster+vectors
base.observe(switch,names='value')
lat=widgets.FloatText(value=50.295,description='Latitude'); lon=widgets.FloatText(value=5.10,description='Longitude')
center=widgets.Button(description='Use coordinates as outlet',button_style='info')
def use_center(_):
    save_geom(mapping(Point(lon.value,lat.value))); widget.center=(lat.value,lon.value); widget.zoom=14
center.on_click(use_center)
display(widgets.VBox([widgets.HTML('<h3>Choose a catchment outlet</h3><p>Navigate using the cached maps. Rivers load automatically for the visible area. Use Refresh rivers to fetch again or Redraw cached to recreate the overlay without a download. Then use the marker tool to place one point on a small stream. Alternatively draw a polygon or rectangle for automatic outlet selection.</p>'),widgets.HBox([base]),widgets.HBox([load_button,redraw_btn,show_rivers,auto_rivers],layout=widgets.Layout(flex_flow='row wrap')),river_status,river_progress,widget,widgets.HBox([lat,lon,center]),status]))

bounds_changed()


In [ ]:
from shapely.geometry import shape, box
import math
assert 'geometry' in selection,'Place a marker or draw a polygon first'
selected=shape(selection['geometry']); OUTLET_POINT=selected if selected.geom_type=='Point' else None
if OUTLET_POINT is not None:
    # 6 km wide initial search area plus the existing 2 km terrain buffer.
    dy=3/111; dx=dy/math.cos(math.radians(selected.y))
    AOI=box(selected.x-dx,selected.y-dy,selected.x+dx,selected.y+dy)
else: AOI=selected
SEARCH_LONLAT=AOI.bounds
assert AOI.is_valid and AOI.area>0
assert 2.8<SEARCH_LONLAT[0]<SEARCH_LONLAT[2]<6.5 and 49.4<SEARCH_LONLAT[1]<SEARCH_LONLAT[3]<51,'Select a small area within Wallonia'
assert SEARCH_LONLAT[2]-SEARCH_LONLAT[0]<.25 and SEARCH_LONLAT[3]-SEARCH_LONLAT[1]<.20
SNAP_MAX_M=100
MAX_CATCHMENT_HA=5000 #@param {type:'integer'}
MIN_CATCHMENT_HA=50
assert MAX_CATCHMENT_HA>=MIN_CATCHMENT_HA
print('Mode:', 'Outlet point' if OUTLET_POINT is not None else 'Automatic AOI','| terrain bounds:',SEARCH_LONLAT)


## 2. Install and prepare model

In [ ]:
import urllib.request
os_release = dict(line.strip().split('=',1) for line in Path('/etc/os-release').read_text().splitlines() if '=' in line)
codename = os_release.get('VERSION_CODENAME','').strip('"')
assert platform.machine() == 'x86_64', 'Expected x86_64 Colab runtime'
assert codename in {'jammy','noble','focal','bookworm'}, f'Unsupported OS for pinned SAGA release: {codename}'
run(['apt-get','update','-qq'])
run(['apt-get','install','-y','git-lfs','saga'])
run(['git','lfs','install'])
def download(url, path):
    if not path.exists():
        tmp = path.with_suffix(path.suffix+'.part')
        urllib.request.urlretrieve(url, tmp)
        tmp.replace(path)
    return path
saga_name = f'saga-vpo_2025.6.11-1-ge95f136.{codename}_amd64.deb'
saga_deb = download('https://github.com/DOV-Vlaanderen/saga-watem/releases/download/2025.6.11/'+saga_name, SOFTWARE/saga_name)
local_deb = Path('/tmp') / saga_name
shutil.copy2(saga_deb, local_deb)
run(['apt-get','install','-y',str(local_deb)])
ws = download('https://github.com/watem-sedem/watem-sedem/releases/download/5.0.3/watem_sedem', SOFTWARE/'watem_sedem_5.0.3')
binary = Path('/usr/local/bin/watem_sedem')
shutil.copy2(ws,binary); binary.chmod(0o755)
os.environ['WATEMSEDEM'] = str(binary.parent)
# saga-vpo contains plugins, not the SAGA command-line program.
# Ubuntu's saga package supplies saga_cmd.
saga_cmd = shutil.which('saga_cmd')
if not saga_cmd:
    run(['dpkg','-L','saga'])
    raise RuntimeError('saga_cmd missing after installing saga')
assert shutil.which('watem_sedem'), 'watem_sedem missing from PATH'
print('SAGA executable:', saga_cmd)
print('WaTEM executable:', shutil.which('watem_sedem'))
plugin_files = subprocess.check_output(['dpkg','-L','saga-vpo'],text=True).splitlines()
print('Installed SAGA plugin libraries:', [p for p in plugin_files if p.endswith('.so')])
run(['saga_cmd','--version'])
for args in (['saga_cmd','topology'], ['saga_cmd','watem']):
    p = subprocess.run(args,capture_output=True,text=True)
    msg = p.stdout+p.stderr
    print(msg[:3500])
    assert 'error: select a library' not in msg.lower() and 'cannot load' not in msg.lower(), f'Missing/incompatible SAGA library: {args}'
run(['ldd',str(binary)])
if not REPO.exists():
    run(['git','clone','https://github.com/watem-sedem/pywatemsedem.git',REPO],
        env={**os.environ,'GIT_LFS_SKIP_SMUDGE':'1'})
run(['git','-C',REPO,'checkout',REV],env={**os.environ,'GIT_LFS_SKIP_SMUDGE':'1'})
run([sys.executable,'-m','pip','install',str(REPO)])
import pywatemsedem
print('pywatemsedem',pywatemsedem.__version__)

In [ ]:
required = ['dtm.tif','K3.tif','basemap_landuse.tif','userchoices.ini']
stems = ['catchm_langegracht','river','infrastructure','roads','water','parcels','grass_strips','buffers','force_routing','outlets']
source = REPO/'tests/data'
names = sorted(p.name for p in source.iterdir() if p.is_file() and (p.name in required or p.stem in stems))
assert set(required).issubset(names)
for stem in stems:
    assert all(stem+ext in names for ext in ('.shp','.shx','.dbf','.prj')), stem
def expected_hash(name):
    raw = subprocess.check_output(['git','-C',str(REPO),'show',f'{REV}:tests/data/{name}'])
    if raw.startswith(b'version https://git-lfs'):
        return next(x.split(':',1)[1] for x in raw.decode().splitlines() if x.startswith('oid sha256:'))
    return hashlib.sha256(raw).hexdigest()
hashes = {name:expected_hash(name) for name in names}
import urllib.request, urllib.error
from concurrent.futures import ThreadPoolExecutor, as_completed

def valid_file(path, name):
    return path.is_file() and hashlib.sha256(path.read_bytes()).hexdigest() == hashes[name]

def valid(folder):
    return all(valid_file(folder/n,n) for n in names)

DATA.mkdir(parents=True,exist_ok=True)
missing = [n for n in names if not valid_file(DATA/n,n)]
print(f'Valid cached files: {len(names)-len(missing)}/{len(names)}')
if missing:
    # Git LFS pull can fail at a detached HEAD or due to upstream LFS service.
    # Capture stderr in the notebook, rather than only showing CalledProcessError.
    pending = [n for n in missing if not valid_file(source/n,n)]
    if pending:
        p = subprocess.run(['git','-C',str(REPO),'lfs','pull','origin',
                            '--include='+','.join('tests/data/'+n for n in pending)],
                           capture_output=True,text=True)
        print('Git LFS return code:',p.returncode)
        print(p.stdout[-4000:])
        print(p.stderr[-6000:])
    def retrieve(name):
        if valid_file(DATA/name,name):
            return name,'cached'
        if valid_file(source/name,name):
            shutil.copy2(source/name,DATA/name)
            return name,'git-lfs'
        # GitHub media endpoint serves actual LFS objects rather than pointers.
        # Always require the exact SHA256 from the pinned upstream Git tree.
        errors=[]
        tmp=DATA/(name+'.part')
        for base in (
            f'https://media.githubusercontent.com/media/watem-sedem/pywatemsedem/{REV}/tests/data/',
            f'https://github.com/watem-sedem/pywatemsedem/raw/{REV}/tests/data/',
            f'https://raw.githubusercontent.com/watem-sedem/pywatemsedem/{REV}/tests/data/',
        ):
            for attempt in range(2):
                try:
                    req=urllib.request.Request(base+name,headers={'User-Agent':'Watem-Colab/1.0'})
                    with urllib.request.urlopen(req,timeout=60) as response, tmp.open('wb') as dst:
                        shutil.copyfileobj(response,dst)
                    if not valid_file(tmp,name):
                        raise ValueError('SHA256 mismatch (possibly a Git LFS pointer)')
                    tmp.replace(DATA/name)
                    return name,'direct download'
                except Exception as e:
                    errors.append(f'{base}: {type(e).__name__}: {e}')
                    tmp.unlink(missing_ok=True)
                    if attempt==0: time.sleep(2)
        raise RuntimeError(name+': '+' | '.join(errors))
    failures=[]
    with ThreadPoolExecutor(max_workers=4) as pool:
        futures={pool.submit(retrieve,name):name for name in missing}
        for f in as_completed(futures):
            try:
                print(*f.result())
            except Exception as e:
                failures.append(str(e))
                print('FAILED:',e)
    if failures:
        raise RuntimeError(f'{len(failures)} data downloads failed. Valid files are retained on Drive. See HTTP/LFS diagnostics above.')
assert valid(DATA), 'Tutorial data checksum validation failed'
(DATA/'source_manifest.json').write_text(json.dumps({'revision':REV,'sha256':hashes},indent=2))
inputdata_folder = WORK/'inputs'
inputdata_folder.mkdir(exist_ok=True)
for n in names: shutil.copy2(DATA/n,inputdata_folder/n)
print(f'Validated {len(names)} files in {DATA}')
print('Local working data:', inputdata_folder)

## 3. Terrain, stream burning, D8 watershed

In [ ]:
import numpy as np, pandas as pd, geopandas as gpd, rasterio, requests, matplotlib.pyplot as plt
from rasterio.warp import reproject, Resampling, transform_bounds
from rasterio.transform import from_origin, xy
from rasterio.features import shapes, rasterize
from shapely.geometry import shape, Point, box
from pysheds.grid import Grid
from IPython.display import display
CRS='EPSG:31370'
bb=transform_bounds('EPSG:4326',CRS,*SEARCH_LONLAT,densify_pts=21)
xmin,ymin=np.floor(np.array(bb[:2])/RES)*RES; xmax,ymax=np.ceil(np.array(bb[2:])/RES)*RES
# buffer around candidate domain; still within tile except possibly west edge; mosaic both adjacent tiles
xmin-=2000; ymin-=2000; xmax+=2000; ymax+=2000
width,height=int((xmax-xmin)/RES),int((ymax-ymin)/RES)
tr=from_origin(xmin,ymax,RES,RES)
profile=dict(driver='GTiff',height=height,width=width,count=1,crs=CRS,transform=tr,nodata=-9999,compress='deflate')
def warp_sources(urls,method):
    dest=np.full((height,width),-9999,dtype='float32')
    for url in urls:
        tmp=np.full_like(dest,-9999)
        with rasterio.open(url) as ds:
            reproject(rasterio.band(ds,1),tmp,dst_transform=tr,dst_crs=CRS,dst_nodata=-9999,resampling=method)
        dest[tmp!=-9999]=tmp[tmp!=-9999]
    assert np.isfinite(dest).all() and (dest!=-9999).all(),'Incomplete raster coverage'
    return dest
def dem_url(lon,lat):
    tile=f'Copernicus_DSM_COG_10_N{lat:02d}_00_E{lon:03d}_00_DEM'
    return f'https://copernicus-dem-30m.s3.amazonaws.com/{tile}/{tile}.tif'
import math
west,south,east,north=transform_bounds(CRS,'EPSG:4326',xmin,ymin,xmax,ymax)
dem_sources=[str(SPW_DEM)] if SPW_DEM else [dem_url(lon,lat) for lon in range(math.floor(west),math.floor(east)+1) for lat in range(math.floor(south),math.floor(north)+1)]
wc_url=[f'https://esa-worldcover.s3.eu-central-1.amazonaws.com/v200/2021/map/ESA_WorldCover_10m_2021_v200_N{lat:02d}E{lon:03d}_Map.tif' for lat in range(math.floor(south/3)*3,math.floor(north/3)*3+1,3) for lon in range(math.floor(west/3)*3,math.floor(east/3)*3+1,3)]
dem=warp_sources(dem_sources,Resampling.bilinear)
lc=warp_sources(wc_url,Resampling.nearest).astype('int16')
def write_raster(path,a,dtype='float32',transform=tr):
    p=profile.copy(); p.update(dtype=dtype,height=a.shape[0],width=a.shape[1],transform=transform)
    with rasterio.open(path,'w',**p) as ds: ds.write(a.astype(dtype),1)
write_raster(WORK/'search_dem.tif',dem)
print('Grid:',dem.shape,'Elevation range:',dem.min(),dem.max(),'Land cover:',np.unique(lc))

In [ ]:
# Retrieve the river network across the FULL buffered DEM region before delineation.
BASE='https://geoservices.wallonie.be/arcgis/rest/services'
sources={'parcels':f'{BASE}/AGRICULTURE/SIGEC_PARC_AGRI_{YEAR}/MapServer/4','soils':f'{BASE}/SOL_SOUS_SOL/CNSW/MapServer/2','roads':f'{BASE}/TOPOGRAPHIE/PICC_VDIFF/MapServer/{PICC_IDS["roads"]}','buildings':f'{BASE}/TOPOGRAPHIE/PICC_VDIFF/MapServer/{PICC_IDS["buildings"]}','river':RIVER_URL,'water':f'{BASE}/TOPOGRAPHIE/PICC_VDIFF/MapServer/{PICC_IDS["water"]}'}
session=requests.Session()
def request_json(url,params):
    response=session.get(url,params=params,timeout=120); response.raise_for_status()
    j=response.json()
    if 'error' in j: raise RuntimeError(j['error'])
    return j
def fetch_features(url, aoi=None):
    aoi=boundary if aoi is None else aoi
    meta=request_json(url,{'f':'json'})
    assert meta.get('geometryType'),f'Not a feature layer: {url}'
    bounds=aoi.total_bounds
    spatial=dict(where='1=1',geometry=','.join(map(str,bounds)),geometryType='esriGeometryEnvelope',inSR=31370,spatialRel='esriSpatialRelIntersects',returnIdsOnly='true',f='json')
    ids=request_json(url+'/query',spatial).get('objectIds') or []
    features=[]
    for i in range(0,len(ids),100):
        batch=request_json(url+'/query',dict(objectIds=','.join(map(str,ids[i:i+100])),outFields='*',outSR=4326,returnGeometry='true',f='geojson'))
        assert not batch.get('exceededTransferLimit'),'Truncated batch'
        features.extend(batch['features'])
    assert len(features)==len(ids),(len(features),len(ids))
    if not features: return gpd.GeoDataFrame(geometry=[],crs=CRS),meta
    g=gpd.GeoDataFrame.from_features(features,crs=4326).to_crs(CRS)
    g.geometry=g.geometry.make_valid()
    return gpd.clip(g,aoi,keep_geom_type=True),meta

search_aoi=gpd.GeoDataFrame(geometry=[box(xmin,ymin,xmax,ymax)],crs=CRS)
river_search,river_meta=fetch_features(sources['river'],search_aoi)
assert river_meta['geometryType']=='esriGeometryPolyline' and len(river_search)>0
river_search=river_search.explode(index_parts=False).reset_index(drop=True)
river_search=river_search[river_search.geometry.geom_type=='LineString'].copy()
assert len(river_search)>0
river_search.to_file(WORK/'river_search.gpkg',driver='GPKG')
print('River features in buffered search area:',len(river_search))
# SAGA native grids via GDAL conversion; stream background must be NoData, not zero.
# Method 0 avoids inventing an authoritative direction from an unoriented vector network.
# A post-fill D8 check then requires mapped river connectivity within selected basin.
BURN_DEPTH=1.0
MIN_SLOPE=0.01
dem_original=dem.copy()
river_cells=rasterize([(g,1) for g in river_search.geometry],out_shape=dem.shape,transform=tr,fill=0,all_touched=True,dtype='uint8').astype(bool)
write_raster(WORK/'stream_mask.tif',np.where(river_cells,1,-9999),'int16')
def saga(*args):
    cmd=['saga_cmd',*map(str,args)]
    proc=subprocess.run(cmd,capture_output=True,text=True,errors='replace')
    print(' '.join(cmd),proc.returncode,flush=True)
    if proc.returncode: print(proc.stdout[-4000:],proc.stderr[-4000:]); proc.check_returncode()
def to_saga(src,dst):
    # Import with SAGA's GDAL tool (preserves NoData and grid geometry).
    saga('io_gdal',0,'-FILES',src,'-GRIDS',dst)
    assert Path(dst).exists(),dst
def from_saga(src,dst):
    saga('io_gdal',2,'-GRIDS',src,'-FILE',dst)
    with rasterio.open(dst) as ds:
        assert ds.shape==dem.shape and ds.transform.almost_equals(tr), (ds.shape,ds.transform,tr)
        return ds.read(1).astype('float32')
to_saga(WORK/'search_dem.tif',WORK/'original.sgrd')
to_saga(WORK/'stream_mask.tif',WORK/'streams.sgrd')
saga('ta_preprocessor',6,'-DEM',WORK/'original.sgrd','-STREAM',WORK/'streams.sgrd','-BURN',WORK/'burned.sgrd','-METHOD',0,'-EPSILON',BURN_DEPTH)
saga('ta_preprocessor',4,'-ELEV',WORK/'burned.sgrd','-FILLED',WORK/'hydro.sgrd','-FDIR',WORK/'hydro_fdir.sgrd','-WSHED',WORK/'hydro_wshed.sgrd','-MINSLOPE',MIN_SLOPE)
dem_burned=from_saga(WORK/'burned.sgrd',WORK/'search_dem_burned.tif')
dem_hydro=from_saga(WORK/'hydro.sgrd',WORK/'search_dem_hydro_export.tif')
assert np.isfinite(dem_hydro).all() and (dem_hydro!=-9999).all()
write_raster(WORK/'search_dem_hydro.tif',dem_hydro)
# Independent steepest D8 slope check, using pysheds default D8 ordering.
# Interior cells only: boundary cells may legitimately drain outside search domain.
dirs=[(-1,0),(-1,1),(0,1),(1,1),(1,0),(1,-1),(0,-1),(-1,-1)]
best=np.full(dem.shape,-np.inf); river_downhill=np.zeros(dem.shape,dtype=bool)
for dr,dc in dirs:
    nbr=np.roll(dem_hydro,(-dr,-dc),axis=(0,1))
    nriv=np.roll(river_cells,(-dr,-dc),axis=(0,1))
    slope=(dem_hydro-nbr)/(RES*np.hypot(dr,dc))
    better=slope>best
    river_downhill[better]=nriv[better] & (slope[better]>0)
    best[better]=slope[better]
interior=np.ones(dem.shape,dtype=bool); interior[[0,-1],:]=False; interior[:,[0,-1]]=False
pits=interior & (best<=0)
assert not pits.any(),f'{pits.sum()} interior cells lack a downhill neighbour'
bad_river=river_cells & interior & ~river_downhill
print('Interior pits/flats:',int(pits.sum()),'River cells without downhill mapped D8 successor (includes natural endpoints):',int(bad_river.sum()))
delta=dem_hydro-dem_original
print('Elevation modification (m): min/median/max',np.min(delta),np.median(delta),np.max(delta))
fig,axes=plt.subplots(1,3,figsize=(16,6),constrained_layout=True)
for ax,a,title in zip(axes,[dem_original,dem_hydro,delta],['Original DEM','Hydrologically conditioned DEM','Elevation modification (m)']):
    im=ax.imshow(a,cmap='terrain' if title!='Elevation modification (m)' else 'RdBu_r')
    ax.contour(river_cells,levels=[.5],colors='blue',linewidths=.3)
    ax.set_title(title); fig.colorbar(im,ax=ax,shrink=.65)
fig.savefig(WORK/'hydro_conditioning.png',dpi=180); plt.show()


In [ ]:
# Compatibility with NumPy >= 2.4: pysheds still uses the removed np.in1d.
# Preserve the old one-dimensional return shape and signature.
if not hasattr(np, 'in1d'):
    def _in1d(ar1, ar2, assume_unique=False, invert=False):
        return np.isin(np.asarray(ar1).ravel(), np.asarray(ar2).ravel(),
                       assume_unique=assume_unique, invert=invert)
    np.in1d = _in1d
    print('Applied pysheds / NumPy in1d compatibility shim.')
grid=Grid.from_raster(str(WORK/'search_dem_hydro.tif'))
z=grid.read_raster(str(WORK/'search_dem_hydro.tif'))
fdir=grid.flowdir(z)
acc=grid.accumulation(fdir)
acc_a=np.asarray(acc)
# candidate outlet within original search bbox (not the surrounding buffer)
rr,cc=np.indices(dem.shape)
xx=xmin+(cc+.5)*RES; yy=ymax-(rr+.5)*RES
upstream_ha=acc_a*RES**2/10000
ORIGINAL_OUTLET=OUTLET_POINT
confirmed_outlet=None
print('Terrain and D8 accumulation ready. Confirm the outlet in the next cell.')


## 4. Confirm the outlet on the model drainage network
Click on a high-accumulation line. The map displays the original point, official river network and DEM drainage. Confirm the selected cell before proceeding.

In [ ]:
# Interactive confirmation on the DEM-derived drainage network; no terrain download.
import ipywidgets as widgets
from ipyleaflet import Map, TileLayer, GeoJSON, CircleMarker, LayersControl, ScaleControl, FullScreenControl, Polyline
from pyproj import Transformer
from shapely.geometry import LineString, mapping
from IPython.display import display
to_ll=Transformer.from_crs(CRS,4326,always_xy=True)
to_xy=Transformer.from_crs(4326,CRS,always_xy=True)
confirmed_outlet=None
outlet_state={'candidate':None,'confirmed':False}
if ORIGINAL_OUTLET is not None:
    cx,cy=to_xy.transform(ORIGINAL_OUTLET.x,ORIGINAL_OUTLET.y)
else:
    cx,cy=to_xy.transform(AOI.centroid.x,AOI.centroid.y)
# Render only high-accumulation D8 cells as short connected vector segments.
# No full raster or opaque low-accumulation background is transferred to the browser.
eligible=(upstream_ha>=MIN_CATCHMENT_HA)&(upstream_ha<=MAX_CATCHMENT_HA)
eligible[[0,-1],:]=False; eligible[:,[0,-1]]=False
offsets={64:(-1,0),128:(-1,1),1:(0,1),2:(1,1),4:(1,0),8:(1,-1),16:(0,-1),32:(-1,-1)}
fd=np.asarray(fdir)
def drainage_geojson(threshold):
    # Vectorized conversion; group short segments into one MultiLineString.
    rr0,cc0=np.where(upstream_ha>=threshold)
    code=fd[rr0,cc0]
    dr=np.zeros(len(rr0),dtype=int); dc=dr.copy(); valid=np.zeros(len(rr0),dtype=bool)
    for k,(dy,dx) in offsets.items():
        hit=code==k; dr[hit]=dy; dc[hit]=dx; valid|=hit
    rt=rr0+dr; ct=cc0+dc
    valid &= (rt>=0)&(rt<fd.shape[0])&(ct>=0)&(ct<fd.shape[1])
    rr0,cc0,rt,ct=[v[valid] for v in (rr0,cc0,rt,ct)]
    x1,y1=to_ll.transform(xx[rr0,cc0],yy[rr0,cc0]); x2,y2=to_ll.transform(xx[rt,ct],yy[rt,ct])
    coordinates=np.round(np.stack([np.column_stack([x1,y1]),np.column_stack([x2,y2])],axis=1),6).tolist()
    return dict(type='FeatureCollection',features=[dict(type='Feature',geometry=dict(type='MultiLineString',coordinates=coordinates),properties={})] if coordinates else []),len(coordinates)
def map_tiles(service,name):
    return TileLayer(url='https://server.arcgisonline.com/ArcGIS/rest/services/'+service+'/MapServer/tile/{z}/{y}/{x}',name=name,attribution='Esri and contributing data providers',max_zoom=20,max_native_zoom=19)
sat=map_tiles('World_Imagery','Satellite'); top=map_tiles('World_Topo_Map','Topographic')
clon,clat=to_ll.transform(cx,cy)
confirm_map=Map(center=(clat,clon),zoom=15,layers=(sat,),scroll_wheel_zoom=True,layout=widgets.Layout(height='580px',width='100%'))
confirm_map.add_control(ScaleControl(position='bottomleft')); confirm_map.add_control(FullScreenControl()); confirm_map.add_control(LayersControl(position='topright'))
# Display the map immediately; send vector overlays only after the widget is displayed.
import asyncio
empty={'type':'FeatureCollection','features':[]}
rhw=None
drain=None
layer_status=widgets.HTML('<b>Preparing river and D8 overlays…</b>')
layer_progress=widgets.IntProgress(value=0,min=0,max=3,description='Map layers:',bar_style='info',layout=widgets.Layout(width='98%'))
overlay_ready=False
overlay_task=None
rhw_data=None
drain_cache={}
if ORIGINAL_OUTLET is not None:
    old=CircleMarker(location=(ORIGINAL_OUTLET.y,ORIGINAL_OUTLET.x),radius=8,color='#ff5252',fill_color='#ff5252',fill_opacity=.9,name='Original point'); confirm_map.add_layer(old)
threshold=widgets.Dropdown(options=[10,25,50,100,250,500],value=50,description='Show ≥ ha')

base_select=widgets.ToggleButtons(options=['Satellite','Topographic'],description='Map:')
def change_base(change):
    confirm_map.layers=((sat if change['new']=='Satellite' else top),)+tuple(l for l in confirm_map.layers if l not in (sat,top))
base_select.observe(change_base,names='value')
feedback=widgets.HTML('Click on a yellow DEM drainage line to select the model outlet.')
confirm_btn=widgets.Button(description='Confirm outlet',button_style='success',disabled=True,icon='check')
suggest_btn=widgets.Button(description='Suggest nearby outlet',button_style='info')
new_marker=None
def choose_candidate(lon,lat,maxdist=40):
    global new_marker,confirmed_outlet
    if not overlay_ready:
        feedback.value='Wait until the drainage layers are ready.'; return
    x,y=to_xy.transform(lon,lat); dist=np.hypot(xx-x,yy-y)
    candidates=np.argwhere(eligible & (dist<=maxdist))
    if not len(candidates):
        feedback.value=f'<b>No eligible drainage cell within {maxdist} m.</b> Click a yellow line with contributing area {MIN_CATCHMENT_HA}–{MAX_CATCHMENT_HA} ha.'; return
    order=np.lexsort((-upstream_ha[candidates[:,0],candidates[:,1]],dist[candidates[:,0],candidates[:,1]]))
    ri,ci=map(int,candidates[order[0]])
    outlet_state.update(candidate=(ri,ci),confirmed=False); confirmed_outlet=None
    lon2,lat2=to_ll.transform(xx[ri,ci],yy[ri,ci])
    if new_marker is not None and new_marker in confirm_map.layers: confirm_map.remove_layer(new_marker)
    new_marker=CircleMarker(location=(lat2,lon2),radius=8,color='#ffffff',fill_color='#20b85a',fill_opacity=1,name='Proposed model outlet'); confirm_map.add_layer(new_marker)
    mapped='on rasterized RHW' if river_cells[ri,ci] else 'off rasterized RHW — D8 model channel will be used'
    shift=float(np.hypot(xx[ri,ci]-cx,yy[ri,ci]-cy))
    feedback.value=f'<b>Proposed outlet:</b> {upstream_ha[ri,ci]:,.1f} ha upstream; click snap {dist[ri,ci]:.1f} m; shift from original/area centre {shift:.1f} m; {mapped}. Click Confirm outlet to proceed.'
    confirm_btn.disabled=False
def click_map(**kwargs):
    if kwargs.get('type')=='click':
        lat,lon=kwargs['coordinates']; choose_candidate(lon,lat)
confirm_map.on_interaction(click_map)
def suggest(_):
    choose_candidate(clon,clat,SNAP_MAX_M)
suggest_btn.on_click(suggest)
def confirm(_):
    global confirmed_outlet
    confirmed_outlet=outlet_state['candidate']; outlet_state['confirmed']=True
    feedback.value='<b style="color:green">Outlet confirmed.</b> Run the next cell to delineate and check the watershed.'
confirm_btn.on_click(confirm)
reload_btn=widgets.Button(description='Refresh / redraw rivers',icon='refresh',layout=widgets.Layout(width='210px'))
async def update_overlays():
    global overlay_ready,rhw_data,rhw,drain
    overlay_ready=False; threshold.disabled=True; reload_btn.disabled=True; suggest_btn.disabled=True; reload_btn.description='Loading map layers…'; reload_btn.icon='spinner'
    layer_progress.value=0; layer_progress.bar_style='info'; layer_progress.layout.display=''
    try:
        layer_status.value='<b>1/3 Preparing official RHW rivers…</b>'
        await asyncio.sleep(.1)
        if rhw_data is None:
            rhw_data=await asyncio.to_thread(lambda:json.loads(river_search[['geometry']].to_crs(4326).to_json()))
        rhw_locations=await asyncio.to_thread(line_locations,rhw_data)
        layer_progress.value=1
        ha=threshold.value
        layer_status.value=f'<b>2/3 Preparing D8 drainage ≥ {ha} ha…</b>'
        if ha not in drain_cache: drain_cache[ha]=await asyncio.to_thread(drainage_geojson,ha)
        data,count=drain_cache[ha]
        layer_progress.value=2
        layer_status.value='<b>3/3 Sending river and drainage overlays to the map…</b>'
        await asyncio.sleep(.1)
        # Recreate both Leaflet polylines, with separate remove/add messages.
        # This avoids reusing stale GeoJSON widget models after changing basemaps.
        drain_locations=await asyncio.to_thread(line_locations,data)
        for old_layer in (rhw,drain):
            if old_layer is not None and old_layer in confirm_map.layers: confirm_map.remove_layer(old_layer)
        await asyncio.sleep(.15)
        rhw=Polyline(locations=rhw_locations,color='#00bfff',weight=3,opacity=1,name='Official RHW rivers',pane='overlayPane',interactive=False)
        drain=Polyline(locations=drain_locations,color='#ffdc00',weight=4,opacity=.95,name='D8 drainage',pane='overlayPane',interactive=False)
        if rhw_locations: confirm_map.add_layer(rhw)
        if drain_locations: confirm_map.add_layer(drain)
        await asyncio.sleep(.3)
        layer_progress.value=3; layer_progress.bar_style='success'
        layer_status.value=f'<b style="color:#218838">✓ Map data ready:</b> {len(rhw_data["features"]):,} RHW features and {count:,} D8 segments sent. If not visible, click Refresh / redraw rivers (no SPW download).'
        overlay_ready=True
    except Exception as exc:
        layer_progress.bar_style='danger'; layer_status.value='Overlay preparation failed: '+str(exc)[:300]
    finally:
        threshold.disabled=False; reload_btn.disabled=False; suggest_btn.disabled=not overlay_ready; reload_btn.description='Refresh / redraw rivers'; reload_btn.icon='refresh'
def schedule_overlay(_=None):
    global overlay_task
    if overlay_task is None or overlay_task.done(): overlay_task=asyncio.ensure_future(update_overlays())
reload_btn.on_click(schedule_overlay)
threshold.observe(schedule_overlay,names='value')
display(widgets.VBox([widgets.HTML('<h3>Confirm the hydrological outlet</h3><p>Blue = official RHW rivers; yellow = DEM-derived high contributing area; red = original point; green = proposed outlet. Click a yellow line and confirm.</p>'),widgets.HBox([base_select,threshold]),widgets.HBox([suggest_btn,reload_btn]),layer_status,layer_progress,confirm_map,feedback,confirm_btn]))
schedule_overlay()


## 5. Delineate the confirmed watershed
If the watershed fails the checks, return to the confirmation map, select another point and rerun this cell. Terrain is already available.

In [ ]:
assert outlet_state['confirmed'] and confirmed_outlet is not None,'Confirm the outlet on the map first.'
candidate=np.array([confirmed_outlet],dtype=int)
print('Candidate outlets:',len(candidate))
chosen=None
for row,col in candidate[:500]:
    m=np.asarray(grid.catchment(x=int(col),y=int(row),fdir=fdir,xytype='index'),dtype=bool)
    if not m.any():
        print('Empty delineation'); continue
    if m[0,:].any() or m[-1,:].any() or m[:,0].any() or m[:,-1].any():
        print('Watershed touches terrain extraction edge; choose a smaller tributary or increase terrain extent before rerunning.'); continue
    area=m.sum()*RES**2/10000
    
    if not (MIN_CATCHMENT_HA<=area<=MAX_CATCHMENT_HA):
        print(f'Outlet drains {area:.1f} ha; allowed range {MIN_CATCHMENT_HA}–{MAX_CATCHMENT_HA} ha.'); continue
    # Mapped river alignment is diagnostic, not a requirement for a valid D8 basin.
    bad=m & river_cells & ~river_downhill
    bad[row,col]=False
    nriver=int((m & river_cells).sum())
    print('Mapped stream alignment:',int(bad.sum()),'of',nriver,'cells lack a mapped D8 successor (including headwater endpoints).')
    # RHW alignment is diagnostic; model channels are derived from D8 accumulation.
    chosen=(int(row),int(col),m); break
assert chosen is not None,'No acceptable basin: inspect the specific rejection reason printed above. Point mode requires the configured area range, complete terrain coverage and D8 drainage closure.'
row,col,mask=chosen
print(f'Confirmed outlet: {upstream_ha[row,col]:.2f} ha upstream; mapped RHW cell: {bool(river_cells[row,col])}')
# Confirm exact D8 closure: every non-outlet pixel has one downstream cell within basin.
D8_OFFSETS={64:(-1,0),128:(-1,1),1:(0,1),2:(1,1),4:(1,0),8:(1,-1),16:(0,-1),32:(-1,-1)}
fd=np.asarray(fdir); rix,cix=np.where(mask); bad=[]
for rr0,cc0 in zip(rix,cix):
    if (rr0,cc0)==(row,col): continue
    off=D8_OFFSETS.get(int(fd[rr0,cc0]))
    if off is None: bad.append((rr0,cc0)); continue
    rt,ct=rr0+off[0],cc0+off[1]
    if not mask[rt,ct] or dem_hydro[rt,ct]>=dem_hydro[rr0,cc0]: bad.append((rr0,cc0))
assert not bad, f'D8 basin has {len(bad)} internal drainage inconsistencies'
assert int(acc_a[row,col])==int(mask.sum()), 'D8 accumulation and basin cell count differ'
print('D8 closure PASS; burned river cells:',int((mask & river_cells).sum()))
misaligned=mask & river_cells & ~river_downhill
misaligned[row,col]=False
fig,ax=plt.subplots(figsize=(9,7)); ax.imshow(np.ma.masked_where(~mask,dem_hydro),cmap='terrain'); ax.contour(mask,levels=[.5],colors='black',linewidths=.7)
r,c=np.where(mask & river_cells); ax.scatter(c,r,s=3,c='blue',label='Mapped stream cells')
r,c=np.where(misaligned); ax.scatter(c,r,s=9,c='orange',label='Mapped cells with non-stream D8 successor')
ax.scatter(col,row,s=40,c='red',label='Outlet'); ax.legend(); ax.set_title('River alignment diagnostic (not D8 closure)'); plt.show()
outxy=xy(tr,row,col)
boundary=gpd.GeoDataFrame(geometry=[shape(g) for g,v in shapes(mask.astype('uint8'),mask=mask,transform=tr) if v==1],crs=CRS).dissolve()
outlet=gpd.GeoDataFrame(geometry=[Point(*outxy)],crs=CRS)
print('Chosen area ha:',mask.sum()*RES**2/10000,'outlet lon/lat:',outlet.to_crs(4326).geometry.iloc[0])
boundary.to_file(PACKAGE/'catchment.shp'); outlet.to_file(PACKAGE/'outlet.gpkg',driver='GPKG')
if OUTLET_POINT is not None:
    gpd.GeoDataFrame(geometry=[OUTLET_POINT],crs=4326).to_crs(CRS).to_file(PACKAGE/'requested_outlet.gpkg',driver='GPKG')
fig,ax=plt.subplots(figsize=(9,8)); boundary.plot(ax=ax,facecolor='#e6e5d8',edgecolor='black'); gpd.clip(river_search,boundary).plot(ax=ax,color='blue',linewidth=2); outlet.plot(ax=ax,color='red',markersize=40); ax.set_title('Selected DEM-derived basin | mapped streams and outlet'); plt.show()
print(f'Agriculture/grass: {100*np.mean(np.isin(lc[mask],[30,40])):.1f}% | D8 area: {mask.sum()*RES**2/10000:.2f} ha')
from ipyleaflet import GeoJSON
bgeo=json.loads(boundary.to_crs(4326).to_json()); ogeo=json.loads(outlet.to_crs(4326).to_json())
confirm_map.add_layer(GeoJSON(data=bgeo,style={'color':'#00e5ff','fillOpacity':.1},name='Delineated watershed'))
confirm_map.add_layer(GeoJSON(data=ogeo,name='Delineated outlet')); display(confirm_map)

## 6. Accept the delineated basin
Inspect the final watershed and agricultural fraction. If unsuitable, select another outlet on the confirmation map, confirm it and rerun the delineation cell. The native model channel raster will be derived from the same conditioned DEM accumulation, using the configurable channel-initiation threshold in the input preparation block.

In [ ]:
ACCEPT=False #@param {type:'boolean'}
assert ACCEPT,'Inspect the watershed and tick ACCEPT before continuing'

## 7. Download data and prepare native model
RHW is used for terrain conditioning and comparison; the final model channel raster comes from D8 accumulation. The model river threshold is configurable.

In [ ]:
vectors={}; metadata={}
for name,url in sources.items():
    print('Fetching',name,flush=True)
    if name=='river':
        vectors[name]=gpd.clip(river_search,boundary,keep_geom_type=True)
        metadata[name]=river_meta
    else:
        vectors[name],metadata[name]=fetch_features(url)
    expected={'parcels':'esriGeometryPolygon','soils':'esriGeometryPolygon','roads':'esriGeometryPolyline','buildings':'esriGeometryPolygon','river':'esriGeometryPolyline','water':'esriGeometryPolygon'}[name]
    assert metadata[name]['geometryType']==expected,(name,metadata[name]['geometryType'])
    if name=='water' and 'NATUR_DESC' in vectors[name]:
        vectors[name]=vectors[name][vectors[name].NATUR_DESC!='Piscine'].copy()
    print(name,len(vectors[name]),list(vectors[name].columns))
    if len(vectors[name]): vectors[name].to_file(PACKAGE/f'{name}_source.gpkg',driver='GPKG')
    (PACKAGE/f'{name}_schema.json').write_text(json.dumps(metadata[name],indent=2))
assert len(vectors['parcels']) and len(vectors['soils'])
display(vectors['parcels'].drop(columns='geometry').head())
display(vectors['soils'].drop(columns='geometry').head())

In [ ]:
p=vectors['parcels'].copy()
fields=metadata['parcels'].get('fields',[])
crop_fields=[f['name'] for f in fields if any(k in (f.get('name','')+' '+f.get('alias','')).lower() for k in ['culture','crop'])]
print('Candidate crop fields:',crop_fields)
# User can set CROP_FIELD explicitly after viewing source schema.
CROP_FIELD='CULT_NOM'
assert CROP_FIELD in p.columns,'Identify official crop attribute in schema'
# Decode coded value domain where provided
field=next(f for f in fields if f['name']==CROP_FIELD)
domain={str(x['code']):x['name'] for x in (field.get('domain') or {}).get('codedValues',[])}
p['crop_label']=p[CROP_FIELD].astype(str).map(lambda x:domain.get(x,x))
def crop_params(label):
    x=label.lower()
    for keys,c,cn,cl in [(['prairie','herbe','grass'],.01,61,'grass'),(['maïs','mais','maize'],.61,81,'maize'),(['pomme de terre','potato'],.64,81,'potato'),(['betterave','beet'],.41,81,'beet'),(['colza','rapeseed'],.38,81,'rape'),(['froment','orge','épeautre','avoine','seigle','triticale','céréale','cereal'],.39,76,'cereal'),(['lin','chanvre'],.68,81,'flax_hemp')]:
        if any(k in x for k in keys): return c,cn,cl
    return .37,81,'UNKNOWN'
p[['C_crop','CN_base','crop_group']]=pd.DataFrame([crop_params(x) for x in p.crop_label],index=p.index)
p['LANDUSE']=np.where(p.crop_group=='grass',-4,-9999).astype(int)
p['C_reduct']=0.0
p['parcel_id']=np.arange(1,len(p)+1)
p['area_ha']=p.geometry.area/10000
display(p.groupby(['crop_label','crop_group','C_crop','CN_base'],dropna=False).area_ha.agg(['count','sum']).reset_index())
p.drop(columns='geometry').to_csv(PACKAGE/'parcel_parameters.csv',index=False)
p.to_file(PACKAGE/'parcels.gpkg',driver='GPKG')
p[['LANDUSE','C_crop','C_reduct','parcel_id','geometry']].to_file(PACKAGE/'parcels.shp')
if (p.crop_group=='UNKNOWN').any(): print('WARNING: unknown crop labels use provisional C/CN; inspect CSV before scientific use.')
# Crop and management tables for future student interface
pd.DataFrame([('baseline',1,0),('CN_only',1,-10),('C_only',.6,0),('combined',.6,-10)],columns=['scenario','C_multiplier','CN_delta']).to_csv(PACKAGE/'management_scenarios.csv',index=False)

In [ ]:
# crop domain subset with 2-cell margin, retaining grid alignment
r0,r1=max(0,np.where(mask)[0].min()-10),min(mask.shape[0],np.where(mask)[0].max()+11)
c0,c1=max(0,np.where(mask)[1].min()-10),min(mask.shape[1],np.where(mask)[1].max()+11)
sl=np.s_[r0:r1,c0:c1]; subtr=tr*rasterio.Affine.translation(c0,r0); m=mask[sl]; cover=lc[sl]
# WaTEM landuse codes: 10 generic crop, -3 forest, -4 pasture, -5 pond, -2 infrastructure
bg=np.full(m.shape,10,dtype='int16')
for k,v in {10:-3,20:-4,30:-4,40:10,50:-2,60:10,70:10,80:-5,90:-4,95:-4,100:-4}.items(): bg[cover==k]=v
bg[~m]=-9999
write_raster(PACKAGE/'dtm_original.tif',dem_original[sl],transform=subtr)
write_raster(PACKAGE/'dtm.tif',dem_hydro[sl],transform=subtr)
write_raster(PACKAGE/'dtm_difference.tif',(dem_hydro-dem_original)[sl],transform=subtr)
write_raster(PACKAGE/'basemap_landuse.tif',bg,'int16',subtr)
write_raster(PACKAGE/'K3.tif',np.full(m.shape,30,dtype='int16'),'int16',subtr)
write_raster(PACKAGE/'HSG_provisional.tif',np.where(m,2,-9999),'int16',subtr)
for name in ['roads','buildings','water']:
    g=vectors[name]
    if len(g):
        g=g[['geometry']].copy()
        if name=='roads': g['paved']=-2
        g.to_file(PACKAGE/(('infrastructure' if name=='buildings' else name)+'.shp'))
# Model river initiation threshold (physical area, independent of display threshold).
MODEL_RIVER_HA=25 #@param {type:'number'}
assert 0<MODEL_RIVER_HA<=float(upstream_ha[row,col]),'Channel threshold must not exceed outlet contributing area'
model_river=mask & (upstream_ha>=MODEL_RIVER_HA)
assert model_river[row,col]
# Every model channel must follow the conditioned D8 network to the chosen outlet.
segments=[]
for ri,ci in zip(*np.where(model_river)):
    if (ri,ci)==(row,col): continue
    dy,dx=D8_OFFSETS[int(fd[ri,ci])]; rt,ct=ri+dy,ci+dx
    assert model_river[rt,ct],'Thresholded D8 channel is discontinuous'
    segments.append(LineString([(xx[ri,ci],yy[ri,ci]),(xx[rt,ct],yy[rt,ct])]))
# Export exact channel raster and GIS line geometry; native code uses raster, not vector re-rasterization.
write_raster(PACKAGE/'model_river.tif',model_river[sl].astype('int16'),'int16',subtr)
write_raster(PACKAGE/'upstream_area_ha.tif',np.where(m,upstream_ha[sl],-9999).astype('float32'),transform=subtr)
if not segments:
    # Isolated outlet is not a useful channel network.
    raise ValueError('Threshold produces only the outlet cell; reduce MODEL_RIVER_HA.')
gpd.GeoDataFrame(geometry=segments,crs=CRS).to_file(PACKAGE/'river.shp')
if len(vectors['river']): vectors['river'][['geometry']].to_file(PACKAGE/'rhw_river.shp')
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
# Categorical palette for the model, not a continuous numerical colour scale.
LC_COLORS=['#fff0a6','#1c6338','#e99b45','#cf3b39','#69bce5','#1769b3','#91ba60']
LC_NAMES=['Cropland','Forest','Grass / pasture','Built-up / roads','Standing water','Model river','Grass strip']
def plot_classes(ax,arr,active,channel=None,title='Land cover'):
    a=np.zeros(arr.shape,dtype='uint8')
    for code,idx in [(-3,1),(-4,2),(-2,3),(-5,4),(-1,5),(-6,6)]: a[arr==code]=idx
    if channel is not None: a[channel]=5
    cmap=ListedColormap(LC_COLORS); cmap.set_bad((0,0,0,0))
    ax.imshow(np.ma.masked_where(~active,a),cmap=cmap,norm=BoundaryNorm(np.arange(-.5,7.5),7),interpolation='nearest')
    ax.set_title(title); ax.set_axis_off()
    ax.legend(handles=[Patch(facecolor=LC_COLORS[i],label=LC_NAMES[i]) for i in range(7) if np.any(active&(a==i))],loc='upper left',bbox_to_anchor=(1.01,1),fontsize=8,frameon=False)
fig,ax=plt.subplots(2,3,figsize=(15,9),constrained_layout=True)
for a,data,title in [(ax[0,0],dem_hydro[sl],'Conditioned elevation (m)'),(ax[0,2],np.where(m,30,np.nan),'Provisional K')]:
    im=a.imshow(np.where(m,data,np.nan)); a.set_title(title); fig.colorbar(im,ax=a,shrink=.7)
plot_classes(ax[0,1],bg,m,model_river[sl],title='Background land cover + model channels')
p.plot(ax=ax[1,0],column='C_crop',legend=True,vmin=0,vmax=1); ax[1,0].set_title('Parcel C factor')
vectors['soils'].plot(ax=ax[1,1],facecolor='none',edgecolor='black'); ax[1,1].set_title('Actual CNSW soil polygons')
boundary.boundary.plot(ax=ax[1,2],color='black')
for name,color in [('roads','grey'),('river','blue'),('buildings','red')]:
    if len(vectors[name]): vectors[name].plot(ax=ax[1,2],color=color)
gpd.GeoDataFrame(geometry=segments,crs=CRS).plot(ax=ax[1,2],color='#00a5ff',linewidth=1.4)
ax[1,2].set_title('Infrastructure, RHW and D8 model channels')
fig.savefig(PACKAGE/'input_atlas.png',dpi=180); plt.show()
manifest=dict(seed=SEED,year=YEAR,area_ha=float(m.sum()*RES**2/10000),outlet_lonlat=list(outlet.to_crs(4326).geometry.iloc[0].coords)[0],resolution=RES,dem=dem_sources,landcover=wc_url,vector_services=sources,K_provisional=30,HSG_provisional='B',unknown_crop_count=int((p.crop_group=='UNKNOWN').sum()),hydro_conditioning={'burn_depth_m':BURN_DEPTH,'min_slope_deg':MIN_SLOPE,'river_source':sources['river'],'method':'SAGA burn method 0 then Wang-Liu fill; D8 closure check'},model_river_threshold_ha=MODEL_RIVER_HA,model_river_cells=int(model_river.sum()),scientific_status='pilot assumptions; not calibrated')
(PACKAGE/'metadata.json').write_text(json.dumps(manifest,indent=2))
print('Saved prepared inputs:',PACKAGE)

In [ ]:
from pywatemsedem.catchment import Catchment
from pywatemsedem.choices import Choices, Options, Parameters, Extensions, ExtensionsParameters, Output
from pywatemsedem.scenario import Scenario
import configparser
assert (PACKAGE/'model_river.tif').exists(),'Prepare D8 model channel raster first.'
catchment=Catchment(SITE_ID,PACKAGE/'catchment.shp',PACKAGE/'dtm.tif',RES,31370,-9999,results_folder=WORK)
catchment.kfactor=PACKAGE/'K3.tif'
catchment.landuse=PACKAGE/'basemap_landuse.tif'
# Supply the exact raster to the wrapper; its setter expects POSITIVE channel IDs
# despite the docstring describing the stored -1 encoding.
rp=catchment.dtm.rp.rasterio_profile
model_grid=np.zeros(catchment.mask.arr.shape,dtype='uint8')
reproject(model_river.astype('uint8'),model_grid,src_transform=tr,src_crs=CRS,dst_transform=rp['transform'],dst_crs=CRS,dst_nodata=0,resampling=Resampling.nearest)
inside0=np.asarray(catchment.mask.arr)==1
assert not np.any((model_grid==1)&~inside0),'Wrapper mask excludes D8 channel cells; do not silently alter channel network'
catchment.river=model_grid.astype('int32')
assert np.array_equal(catchment.river.arr==-1,model_grid==1),'Wrapper changed the supplied river raster'
catchment.kfactor.arr[:]=30
print('D8 model channel cells:',int(model_grid.sum()),'| threshold:',MODEL_RIVER_HA,'ha')
for prop,name in [('vct_infrastructure_buildings','infrastructure'),('vct_infrastructure_roads','roads'),('vct_water','water')]:
    path=PACKAGE/f'{name}.shp'
    if path.exists(): setattr(catchment,prop,path)
default_choices=inputdata_folder/'userchoices.ini'
def settings():
    o=Options(); o.read_values_from_ini(default_choices)
    par=Parameters(); par.read_values_from_ini(default_choices)
    ext=Extensions(); ext.read_values_from_ini(default_choices)
    out=Output(); out.read_values_from_ini(default_choices)
    ep=ExtensionsParameters(ext); ep.apply_defaults()
    for v in (ep.ktc_low,ep.ktc_high,ep.ktc_limit): v.read_value_from_ini(default_choices)
    return Choices(o,par,ext,ep,out)
rain=np.array([0,2,4,7,9,8,5,3,2,0],dtype=float)
rainfile=WORK/'storm.txt'; np.savetxt(rainfile,np.column_stack([np.arange(len(rain))*10,rain]),fmt=['%d','%.3f'],delimiter='\t')
intensity=rain[1:]*6; R=float(np.sum(.29*(1-.72*np.exp(-.05*intensity))*rain[1:])*np.max(np.convolve(rain[1:],np.ones(3),'valid'))*2)
scenarios={}
for nr,mode in enumerate(['annual','event'],1):
    s=Scenario(catchment,YEAR,nr,settings())
    # Raster-only river network; no segment topology or routing tables are supplied.
    s.choices.extensions.river_routing=False
    if mode=='event':
        s.choices.extensions.curve_number=True; s.choices.extensions.convert_output=False
        s.choices.parameters.r_factor=R
        ep=s.choices.extensionparameters; ep.alpha=0.; ep.beta=0.; ep.antecedent_rainfall=0.; ep.stream_velocity=.2; ep.desired_timestep=60; ep.endtime_model=720
        s.choices.output.write_total_runoff=True; s.choices.output.write_rainfall_excess=True
    s.vct_parcels=PACKAGE/'parcels.shp'
    s.composite_landuse=s.create_composite_landuse(); s.cfactor=s.create_cfactor()
    assert np.array_equal(s.composite_landuse.arr==-1,model_grid==1),'Composite land use changed model channel cells'
    assert np.all(s.cfactor.arr[model_grid==1]==0),'Channel C factor must be zero'
    ep=s.choices.extensionparameters
    s.ktc=s.create_ktc(ep.ktc_low.value,ep.ktc_high.value,ep.ktc_limit.value,not s.choices.extensions.create_ktc_map.value)
    if mode=='event':
        lu=np.asarray(s.composite_landuse.arr); inside=np.asarray(catchment.mask.arr)==1
        cn=np.zeros(lu.shape,dtype='float32')
        cn[inside & (lu>0)]=81
        # Use model grid profile, not the source raster shape.
        pr=s.rp.rasterio_profile
        crop_cn=rasterize([(g,float(v)) for g,v in zip(p.geometry,p.CN_base)],out_shape=lu.shape,transform=pr['transform'],fill=0,dtype='float32')
        use=inside & (lu>0)&(crop_cn>0); cn[use]=crop_cn[use]
        for k,v in {-1:100,-2:98,-3:55,-4:61,-5:100,-6:61}.items(): cn[inside & (lu==k)]=v
        assert np.all((cn[inside]>0)&(cn[inside]<=100)),np.unique(lu[inside & (cn==0)])
        s.cn=cn; s.rainfall=rainfile
    # Explicitly refill the entire wrapper rectangular DEM from the full conditioned search grid.
    from rasterio.warp import reproject, Resampling
    model_profile=s.rp.rasterio_profile
    full=np.full(s.composite_landuse.arr.shape,np.nan,dtype='float32')
    reproject(dem_hydro,full,src_transform=tr,src_crs=CRS,dst_transform=model_profile['transform'],dst_crs=CRS,dst_nodata=np.nan,resampling=Resampling.nearest)
    assert np.isfinite(full).all() and (full>-1000).all(), 'Model rectangular window lacks valid DEM coverage'
    catchment.dtm.arr=full.copy()
    s.prepare_input_files(); s.create_ini_file()
    # Verify actual exported DTM, PRC and river pixels before running.
    native_dir=s.sfolder.wsinput_folder
    ini_check=configparser.ConfigParser(interpolation=None); ini_check.read(s.ini)
    dtm_candidates=[v for sec in ini_check.sections() for k,v in ini_check[sec].items() if 'dtm' in k.lower() and v.lower().endswith(('.rst','.tif'))]
    assert len(dtm_candidates)==1,dtm_candidates
    dtm_path=native_dir/Path(dtm_candidates[0]).name
    with rasterio.open(dtm_path) as ds:
        before=ds.read(1,masked=True)
        native_transform=ds.transform
        native_crs=ds.crs
        print('Wrapper DEM before repair:', 'nodata',ds.nodata,'masked',int(np.ma.getmaskarray(before).sum()),'raw range',float(np.min(before.data)),float(np.max(before.data)))
        assert ds.shape==full.shape and native_transform.almost_equals(model_profile['transform']), 'Native grid alignment mismatch'
    # Write directly using GDAL RST CreateCopy via rasterio, including a new RDC.
    # Verify the round trip and grid alignment before replacing native files.
    from rasterio.shutil import copy as rio_copy
    tif=WORK/f'{mode}_full_dtm.tif'
    with rasterio.open(tif,'w',driver='GTiff',height=full.shape[0],width=full.shape[1],count=1,dtype='float32',crs=CRS,transform=model_profile['transform'],nodata=-99999.) as ds:
        ds.write(full,1)
    temp=WORK/f'{mode}_full_dtm.rst'
    rio_copy(tif,temp,driver='RST')
    with rasterio.open(temp) as ds:
        raw=ds.read(1,masked=True)
        assert ds.dtypes[0]=='float32' and not np.ma.getmaskarray(raw).any() and np.allclose(raw.data,full,atol=.001), 'Direct RST creation failed'
        assert ds.transform.almost_equals(native_transform)
    # Replace both raster and header; delete stale GDAL auxiliary metadata.
    for suffix in ['.rst','.rdc']:
        shutil.copy2(temp.with_suffix(suffix),dtm_path.with_suffix(suffix))
    for suffix in ['.aux.xml','.msk','.ovr']:
        Path(str(dtm_path)+suffix).unlink(missing_ok=True)
    with rasterio.open(dtm_path) as ds:
        native_dem=ds.read(1,masked=True)
        print('Repaired native DEM:',ds.shape,ds.dtypes[0],'nodata',ds.nodata,'masked',int(np.ma.getmaskarray(native_dem).sum()),'range',float(native_dem.data.min()),float(native_dem.data.max()))
        assert ds.transform.almost_equals(native_transform) and ds.shape==full.shape
        assert not np.ma.getmaskarray(native_dem).any() and np.isfinite(native_dem.data).all() and (native_dem.data>-1000).all(), 'Native DEM contains invalid elevations'
        assert np.allclose(native_dem.data,full,atol=.001), 'Native DEM differs from full window'
    lu=np.asarray(s.composite_landuse.arr); inside=np.asarray(catchment.mask.arr)==1
    print(mode,'model cells',inside.sum(),'river cells',np.sum(inside & (lu==-1)),'DEM range',float(full.min()),float(full.max()))
    assert np.any(inside & (lu==-1)), 'No native river cells in basin'
    print('Outside-domain parcel codes:',np.unique(lu[~inside]))
    # Native executable recognizes only PRC=0 as outside; -9999 is NOT an outside code.
    ini_files={k:v for sec in ini_check.sections() for k,v in ini_check[sec].items() if v.lower().endswith('.rst')}
    print('INI raster references:',ini_files)
    def native_read(key):
        matches=[v for k,v in ini_files.items() if key(k)]
        assert len(matches)==1,matches
        with rasterio.open(native_dir/Path(matches[0]).name) as ds: return ds.read(1),ds.transform
    prc_matches=[v for k,v in ini_files.items() if 'parcel' in k.lower() and 'filename' in k.lower()]
    if len(prc_matches)!=1:
        prc_matches=[f.name for f in native_dir.glob('*.rst') if 'parcelmosaic' in f.stem.lower() or f.stem.lower() in ('prc','perceelskaart')]
    assert len(prc_matches)==1,('Cannot locate executable parcel raster',ini_files)
    prc_path=native_dir/Path(prc_matches[0]).name
    with rasterio.open(prc_path) as ds: native_prc=ds.read(1); prc_transform=ds.transform
    print('Native PRC outside raw codes BEFORE:',np.unique(native_prc[~inside]),'inside codes:',np.unique(native_prc[inside]))
    assert prc_transform.almost_equals(native_transform)
    # Rebuild native PRC from actual model array and explicit outside=0, not NoData.
    prc=np.where(inside,lu,0).astype('int16')
    assert np.all((prc[inside]>0)|(np.isin(prc[inside],[-1,-2,-3,-4,-5,-6])))
    assert np.max(prc)<=32757,'Parcel IDs exceed model range'
    prctif=WORK/f'{mode}_prc.tif'; prcrst=WORK/f'{mode}_prc.rst'
    with rasterio.open(prctif,'w',driver='GTiff',height=prc.shape[0],width=prc.shape[1],count=1,dtype='int16',crs=CRS,transform=native_transform,nodata=-9999) as ds: ds.write(prc,1)
    rio_copy(prctif,prcrst,driver='RST')
    for suffix in ['.rst','.rdc']: shutil.copy2(prcrst.with_suffix(suffix),prc_path.with_suffix(suffix))
    for suffix in ['.aux.xml','.msk','.ovr']: Path(str(prc_path)+suffix).unlink(missing_ok=True)
    with rasterio.open(prc_path) as ds:
        native_prc=ds.read(1); assert np.array_equal(native_prc,prc)
    print('Native PRC outside raw codes AFTER:',np.unique(native_prc[~inside]),'native river cells:',int((native_prc==-1).sum()))
    assert np.all(native_prc[~inside]==0)
    assert np.array_equal(native_prc==-1,model_grid==1),'Native PRC changed the D8 model river raster'
    rr_out,cc_out=rasterio.transform.rowcol(native_transform,*outxy)
    assert native_prc[rr_out,cc_out]==-1,'Confirmed outlet is not a native river cell'
    # Constant provisional K is defined on the entire model grid, not masked by
    # a separately rasterized watershed polygon. PRC alone defines active cells.
    k_matches=[v for k,v in ini_files.items() if k.lower()=='k factor filename']
    assert len(k_matches)==1,k_matches
    kpath=native_dir/Path(k_matches[0]).name
    with rasterio.open(kpath) as ds:
        k_before=ds.read(1)
        assert ds.shape==full.shape and ds.transform.almost_equals(native_transform)
    invalid_k=inside & (~np.isfinite(k_before) | (k_before<0))
    print(mode,'K QA: invalid active cells before native normalization:',int(invalid_k.sum()))
    # Quantify the original D8 mask versus wrapper mask before making repairs.
    d8_grid=np.zeros(inside.shape,dtype='uint8')
    reproject(mask.astype('uint8'),d8_grid,src_transform=tr,src_crs=CRS,dst_transform=native_transform,dst_crs=CRS,dst_nodata=0,resampling=Resampling.nearest)
    print('D8 vs wrapper mask: D8 cells',int(d8_grid.sum()),'wrapper cells',int(inside.sum()),'wrapper-only',int((inside & (d8_grid==0)).sum()),'D8-only',int(((d8_grid==1)&~inside).sum()))
    if invalid_k.any():
        print('WARNING: wrapper export introduced invalid K; full-grid constant normalization follows.')
    k_fixed=np.full(full.shape,30,dtype='int16')
    ktif=WORK/f'{mode}_k_full.tif'; krst=WORK/f'{mode}_k_full.rst'
    with rasterio.open(ktif,'w',driver='GTiff',height=full.shape[0],width=full.shape[1],count=1,dtype='int16',crs=CRS,transform=native_transform,nodata=-9999) as ds: ds.write(k_fixed,1)
    rio_copy(ktif,krst,driver='RST')
    for suffix in ['.rst','.rdc']: shutil.copy2(krst.with_suffix(suffix),kpath.with_suffix(suffix))
    for suffix in ['.aux.xml','.msk','.ovr']: Path(str(kpath)+suffix).unlink(missing_ok=True)
    with rasterio.open(kpath) as ds:
        k_after=ds.read(1,masked=True)
        assert ds.transform.almost_equals(native_transform) and ds.dtypes[0]=='int16'
        assert not np.ma.getmaskarray(k_after).any() and np.array_equal(k_after.data,k_fixed)
    print(mode,'K repair: all grid cells = 30 kg h/(MJ mm); invalid active cells = 0')
    # Verify raw numerical parameter values in the active model domain.
    for k,v in ini_files.items():
        if not any(x in k.lower() for x in ['c factor','c-factor','k factor','k-factor','p factor','p-factor','ktc']): continue
        with rasterio.open(native_dir/Path(v).name) as ds:
            a=ds.read(1); vv=a[inside & (native_prc!=-1)]
            print(k,'active min/max',float(vv.min()),float(vv.max()),'negative cells',int((vv<0).sum()))
            assert np.isfinite(vv).all() and (vv>=0).all(),f'Invalid active {k} values'
            if 'p factor' in k.lower() or 'p-factor' in k.lower(): assert (vv>0).all(),'P=0 triggers invalid RUSLE sentinel in Pascal 5.0.3'
    if mode=='annual':
        fig,ax=plt.subplots(figsize=(9,7)); plot_classes(ax,native_prc,inside,title='Final model land cover (PRC)'); fig.savefig(PACKAGE/'model_landcover.png',dpi=180,bbox_inches='tight'); plt.show()
    np.savez_compressed(WORK/f'{mode}_native_domain.npz',prc=native_prc,mask=inside,dem=full)
    assert not np.any(lu[~inside]>0), 'Agricultural parcel IDs outside model domain'
    if mode=='event':
        cfg=configparser.ConfigParser(interpolation=None); cfg.optionxform=str; cfg.read(s.ini)
        sec=next(x for x in cfg.sections() if x.lower()=='parameters extensions')
        for key,value in [('Endtime model',720),('Bulk density',int(s.choices.parameters.bulk_density.value))]:
            actual=next((x for x in cfg[sec] if x.lower()==key.lower()),key); cfg[sec][actual]=str(value)
        with s.ini.open('w') as f: cfg.write(f)
    scenarios[mode]=s
    print(mode,s.ini)
DEST=OUTPUT/RUN_ID; DEST.mkdir(exist_ok=True)
for mode,s in scenarios.items(): shutil.copytree(s.sfolder.scenario_folder,DEST/mode,dirs_exist_ok=True)
print('Native inputs copied; INI paths refer to current local runtime. Regenerate on next runtime.')

## 6. Run and report baseline

In [ ]:
# Run baseline annual and 40 mm event using the verified native inputs.
import urllib.request, sys
helper_dir=Path('/content/watem_student_helpers'); helper_dir.mkdir(exist_ok=True)
urllib.request.urlretrieve('https://raw.githubusercontent.com/kvanoost/Watem/50839661b3a72d2f7a78b7c3d0a4ba7019908cc7/watem_student_common.py',helper_dir/'watem_student_common.py')
sys.path.insert(0,str(helper_dir))
import watem_student_common as ws

BASELINE=SITE/'baseline'; BASELINE.mkdir(exist_ok=True)
rows=[]
for mode,s in scenarios.items():
    dest=BASELINE/('annual' if mode=='annual' else 'event_40mm')
    ws.clone_run(s.sfolder.scenario_folder,dest,binary,40 if mode=='event' else None)
    ws.execute(dest,binary)
    rows.append(ws.report(dest,dest.name,40 if mode=='event' else None))
# Two additional storms, same duration and shape.
for depth in (20,60):
    dest=BASELINE/f'event_{depth}mm'
    ws.clone_run(BASELINE/'event_40mm',dest,binary,depth); ws.execute(dest,binary)
    rows.append(ws.report(dest,dest.name,depth))
display(pd.DataFrame(rows)); pd.DataFrame(rows).to_csv(BASELINE/'summary.csv',index=False)
(BASELINE/'complete.json').write_text(json.dumps(dict(kind='baseline',baseline_id=ws.baseline_digest(BASELINE),site=SITE_ID,year=YEAR,resolution=RES,storms=[20,40,60],parameter_status='provisional K=30, HSG B, annual C used for event',reference=str(PACKAGE)),indent=2))


## 7. Download GIS package

In [ ]:
# Empty shapefile templates with explicit schema, readable in ArcGIS and QGIS.
import fiona
from google.colab import files
GIS=WORK/'GIS_baseline'; GIS.mkdir(exist_ok=True)
for sub in ['reference','baseline_results','editable_scenario']: (GIS/sub).mkdir()
for f in PACKAGE.iterdir():
    if f.is_file() and f.suffix.lower() in ['.shp','.shx','.dbf','.prj','.cpg','.gpkg','.tif','.csv','.json']: shutil.copy2(f,GIS/'reference'/f.name)
for mode in ['annual','event_20mm','event_40mm','event_60mm']: shutil.copytree(BASELINE/mode/'report',GIS/'baseline_results'/mode)
schemas={'C_change':('Polygon',{'id':'int','c_mult':'float','practice':'str:80','years':'int','comment':'str:200'}),'CN_change':('Polygon',{'id':'int','cn_delta':'float','soc_base':'float','soc_new':'float','years':'int','practice':'str:80','comment':'str:200'}),'grass':('LineString',{'id':'int','width_m':'float','type':'str:40','comment':'str:200'})}
for name,(geom,props) in schemas.items():
    with fiona.open(GIS/'editable_scenario'/f'{name}.shp','w',driver='ESRI Shapefile',crs='EPSG:31370',schema={'geometry':geom,'properties':props}): pass
(GIS/'README.txt').write_text('''Open layers in ArcGIS Pro or QGIS. Edit ONLY the three shapefile templates. Draw polygons for C multiplier (0–1) and CN delta (-20 to 0); draw grass centerlines with total width in metres. Save edits; ZIP the editable_scenario folder including SHP, SHX, DBF, PRJ. Upload in NB02. SOC fields document assumptions only. Parameters are provisional; compare relative effects, not validated predictions. River outlet discharge is the model-reported automatic outlet. Annual C is reused for events in this pilot.''')
# A single portable ZIP contains GIS layers and machine-readable native baseline inputs.
shutil.copytree(BASELINE,GIS/'baseline',dirs_exist_ok=True)
archive=shutil.make_archive(str(SITE/f'{SITE_ID}_Baseline'),'zip',GIS)
print('Save this ZIP locally: it is needed for NB02 and NB03.'); files.download(archive)
